In [4]:
import cudf
print(f"cuDF version: {cudf.__version__}")

df_test = cudf.DataFrame({'a': [1, 2, 3]})
print(df_test * 2)
print("cuDF: 利用可能!")

cuDF version: 26.04.00
   a
0  2
1  4
2  6
cuDF: 利用可能!


In [1]:
import polars as pl
import cudf
import pandas as pd
import numpy as np
import json
import time
import gc
from pathlib import Path
from datetime import date, timedelta
from collections import defaultdict

DATA_DIR = Path('/workspace/data/raw')
CANDIDATE_DIR = Path('/workspace/outputs/candidates')
FEATURE_DIR = Path('/workspace/outputs/features')
MODEL_DIR = Path('/workspace/outputs/models')
CONVERTED_DIR = Path('/workspace/outputs/converted')
for d in [FEATURE_DIR, MODEL_DIR, CANDIDATE_DIR, CONVERTED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EVAL_CUTOFFS = [12, 24, 48, 96, 120]
TRAIN_CUTOFF = date(2020, 9, 8)
VALID_CUTOFF = date(2020, 9, 15)
N_CANDIDATES = 100

# 整数変換済みデータ読み込み
map_path = CONVERTED_DIR / 'id_maps.json'
with open(map_path, 'r') as f:
    maps = json.load(f)
customer_id_map = maps['customer_id_map']
customer_id_reverse = {int(v): k for k, v in customer_id_map.items()}
article_id_map = maps['article_id_map']
article_id_reverse = {int(v): k for k, v in article_id_map.items()}

transactions = pl.read_parquet(CONVERTED_DIR / 'transactions.parquet')
customers = pl.read_parquet(CONVERTED_DIR / 'customers.parquet')
articles = pl.read_parquet(CONVERTED_DIR / 'articles.parquet')

# cuDFに変換
transactions_gdf = cudf.from_pandas(transactions.to_pandas())
customers_gdf = cudf.from_pandas(customers.to_pandas())
articles_gdf = cudf.from_pandas(articles.to_pandas())

# best_weights読み込み
with open(CANDIDATE_DIR / 'best_weights_2020-09-15.json', 'r') as f:
    weights_data = json.load(f)
best_weights = weights_data['weights']
active_logics = {k: v for k, v in best_weights.items() if v > 0}

print(f"transactions: {len(transactions):,}行")
print(f"使用ロジック: {list(active_logics.keys())}")
print("セル1完了!")

transactions: 31,788,324行
使用ロジック: ['item2item_cf', 'bin_popular', 'purchase_interval', 'timedecay', 'repurchase', 'variant', 'also_bought']
セル1完了!


In [2]:
# 保存先を変更
FEATURE_DIR = Path('/home/rapids/features')
MODEL_DIR = Path('/home/rapids/models')
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"FEATURE_DIR: {FEATURE_DIR}")
print(f"MODEL_DIR: {MODEL_DIR}")


FEATURE_DIR: /home/rapids/features
MODEL_DIR: /home/rapids/models


In [3]:
# ============================================================
# 共通関数
# ============================================================
def get_actuals(transactions, cutoff, label_days=7):
    label_end = cutoff + timedelta(days=label_days)
    grouped = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(cutoff)) & (pl.col('t_dat') <= pl.lit(label_end)))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('actual'))
    )
    return {row['customer_id']: set(row['actual']) for row in grouped.iter_rows(named=True)}

def apk(actual_set, predicted, k):
    if not actual_set:
        return 0.0
    score = 0.0
    hits = 0
    seen = set()
    for rank, article_id in enumerate(predicted[:k], start=1):
        if article_id in seen:
            continue
        seen.add(article_id)
        if article_id in actual_set:
            hits += 1
            score += hits / rank
    return score / min(len(actual_set), k)

def load_candidates(name, cutoff):
    path = CANDIDATE_DIR / f"{name}_{cutoff}.parquet"
    if path.exists():
        return pl.read_parquet(path)
    return None

def create_candidates_scored(cutoff, active_logics, valid_customers=None):
    scored_list = []
    for name, w in active_logics.items():
        df = load_candidates(name, cutoff)
        if df is None:
            continue
        if valid_customers is not None:
            df = df.filter(pl.col('customer_id').is_in(valid_customers))
        normalized = (
            df.select(['customer_id', 'article_id', 'score'])
            .join(
                df.group_by('customer_id').agg([
                    pl.col('score').max().alias('max_score'),
                    pl.col('score').min().alias('min_score'),
                ]),
                on='customer_id', how='left'
            )
            .with_columns(
                pl.when(pl.col('max_score') == pl.col('min_score'))
                .then(1.0)
                .otherwise(
                    (pl.col('score') - pl.col('min_score')) /
                    (pl.col('max_score') - pl.col('min_score'))
                )
                .alias('norm_score')
            )
            .with_columns((pl.col('norm_score') * w).alias('weighted_score'))
            .select(['customer_id', 'article_id', 'weighted_score'])
        )
        scored_list.append(normalized)
    combined = pl.concat(scored_list)
    return (
        combined
        .group_by(['customer_id', 'article_id'])
        .agg(pl.col('weighted_score').sum().alias('candidate_score'))
        .with_columns(
            pl.col('candidate_score').rank(method='ordinal', descending=True)
            .over('customer_id').alias('rank')
        )
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )

# ============================================================
# 特徴量作成関数（キャッシュ付き）
# ============================================================
def build_user_features(transactions_gdf, customers_gdf, cutoff):
    """ユーザーベース特徴量（cuDF版・高速化）"""
    # キャッシュ確認
    cache_path = FEATURE_DIR / f'user_features_{cutoff}.parquet'
    if cache_path.exists():
        print(f"  ユーザー特徴量: キャッシュ読み込み {cache_path}")
        return cudf.read_parquet(str(cache_path))
    
    print("  ユーザー特徴量作成中...")
    start = time.time()
    cutoff_dt = np.datetime64(cutoff)
    df = transactions_gdf[transactions_gdf['t_dat'] <= cutoff_dt]
    
    user_basic = df.groupby('customer_id').agg({
        'article_id': ['count', 'nunique'],
        't_dat': ['max', 'min']
    })
    user_basic.columns = ['total_purchases', 'unique_articles', 'last_purchase_date', 'first_purchase_date']
    user_basic = user_basic.reset_index()
    user_basic['days_since_last_purchase'] = (cutoff_dt - user_basic['last_purchase_date']).dt.days
    user_basic['days_since_first_purchase'] = (cutoff_dt - user_basic['first_purchase_date']).dt.days
    user_basic = user_basic.drop(columns=['last_purchase_date', 'first_purchase_date'])
    
    for days in [7, 14, 30, 60]:
        start_dt = np.datetime64(cutoff - timedelta(days=days))
        recent = df[df['t_dat'] >= start_dt]
        recent_count = recent.groupby('customer_id').agg({'article_id': 'count'})
        recent_count.columns = [f'purchases_last_{days}d']
        recent_count = recent_count.reset_index()
        user_basic = user_basic.merge(recent_count, on='customer_id', how='left')
        user_basic[f'purchases_last_{days}d'] = user_basic[f'purchases_last_{days}d'].fillna(0)
    
    start_30 = np.datetime64(cutoff - timedelta(days=30))
    recent30 = df[df['t_dat'] >= start_30]
    recent30_unique = recent30.groupby('customer_id').agg({'article_id': 'nunique'})
    recent30_unique.columns = ['unique_articles_last_30d']
    recent30_unique = recent30_unique.reset_index()
    user_basic = user_basic.merge(recent30_unique, on='customer_id', how='left')
    user_basic['unique_articles_last_30d'] = user_basic['unique_articles_last_30d'].fillna(0)
    
    user_basic['purchase_frequency_per_week'] = (
        user_basic['total_purchases'] / (user_basic['days_since_first_purchase'] / 7.0).clip(lower=1)
    )
    
    basket = df.groupby(['customer_id', 't_dat']).agg({'article_id': 'count'})
    basket.columns = ['basket_size']
    basket = basket.reset_index()
    avg_basket = basket.groupby('customer_id').agg({'basket_size': 'mean'})
    avg_basket.columns = ['avg_basket_size']
    avg_basket = avg_basket.reset_index()
    user_basic = user_basic.merge(avg_basket, on='customer_id', how='left')
    user_basic['avg_basket_size'] = user_basic['avg_basket_size'].fillna(1.0)
    
    ua_count = df.groupby(['customer_id', 'article_id']).agg({'t_dat': 'count'})
    ua_count.columns = ['purchase_count']
    ua_count = ua_count.reset_index()
    ua_count['is_repeat'] = (ua_count['purchase_count'] > 1).astype('int32')
    repeat_stats = ua_count.groupby('customer_id').agg({'is_repeat': 'mean'})
    repeat_stats.columns = ['repeat_rate']
    repeat_stats = repeat_stats.reset_index()
    user_basic = user_basic.merge(repeat_stats, on='customer_id', how='left')
    user_basic['repeat_rate'] = user_basic['repeat_rate'].fillna(0)
    
    hist_before = df[df['t_dat'] < start_30]
    hist_before_set = hist_before[['customer_id', 'article_id']].drop_duplicates()
    hist_before_set['is_old'] = 1
    recent30_all = df[df['t_dat'] >= start_30][['customer_id', 'article_id']].drop_duplicates()
    recent30_merged = recent30_all.merge(hist_before_set, on=['customer_id', 'article_id'], how='left')
    recent30_merged['is_old'] = recent30_merged['is_old'].fillna(0)
    recent30_merged['is_new'] = 1 - recent30_merged['is_old']
    new_rate = recent30_merged.groupby('customer_id').agg({'is_new': 'mean'})
    new_rate.columns = ['new_article_rate_30d']
    new_rate = new_rate.reset_index()
    user_basic = user_basic.merge(new_rate, on='customer_id', how='left')
    user_basic['new_article_rate_30d'] = user_basic['new_article_rate_30d'].fillna(0)
    
    articles_gdf_slim = cudf.read_parquet(str(CONVERTED_DIR / 'articles.parquet'),
                                          columns=['article_id', 'product_type_no'])
    df_with_pt = df.merge(articles_gdf_slim, on='article_id', how='left')
    cat_div = df_with_pt.groupby('customer_id').agg({'product_type_no': 'nunique'})
    cat_div.columns = ['category_diversity']
    cat_div = cat_div.reset_index()
    user_basic = user_basic.merge(cat_div, on='customer_id', how='left')
    user_basic['category_diversity'] = user_basic['category_diversity'].fillna(0)
    
    df_sorted = df[['customer_id', 't_dat']].sort_values(['customer_id', 't_dat'])
    df_sorted['prev_t_dat'] = df_sorted.groupby('customer_id')['t_dat'].shift(1)
    df_sorted = df_sorted.dropna(subset=['prev_t_dat'])
    df_sorted['interval_days'] = (df_sorted['t_dat'] - df_sorted['prev_t_dat']).dt.days
    avg_interval = df_sorted.groupby('customer_id').agg({'interval_days': 'mean'})
    avg_interval.columns = ['avg_purchase_interval']
    avg_interval = avg_interval.reset_index()
    user_basic = user_basic.merge(avg_interval, on='customer_id', how='left')
    user_basic['avg_purchase_interval'] = user_basic['avg_purchase_interval'].fillna(-1)
    
    user_features = user_basic.merge(
        customers_gdf[['customer_id', 'age', 'age_bin', 'FN', 'Active',
                        'club_member_status_enc', 'fashion_news_frequency_enc']],
        on='customer_id', how='left'
    )
    
    # 保存
    user_features.to_pandas().to_parquet(str(cache_path))
    elapsed = time.time() - start
    print(f"  ユーザー特徴量: {user_features.shape} （{elapsed/60:.1f}分）保存完了")
    return user_features


def build_article_features(transactions_gdf, articles_gdf, cutoff):
    """商品ベース特徴量（cuDF版）"""
    cache_path = FEATURE_DIR / f'article_features_{cutoff}.parquet'
    if cache_path.exists():
        print(f"  商品特徴量: キャッシュ読み込み {cache_path}")
        return cudf.read_parquet(str(cache_path))
    
    print("  商品特徴量作成中...")
    start = time.time()
    cutoff_dt = np.datetime64(cutoff)
    df = transactions_gdf[transactions_gdf['t_dat'] <= cutoff_dt]
    
    article_basic = df.groupby('article_id').agg({
        'customer_id': ['count', 'nunique'],
        't_dat': 'min'
    })
    article_basic.columns = ['total_sales', 'unique_buyers', 'first_sale_date']
    article_basic = article_basic.reset_index()
    article_basic['days_since_first_sale'] = (cutoff_dt - article_basic['first_sale_date']).dt.days
    article_basic = article_basic.drop(columns=['first_sale_date'])
    
    for days in [7, 14, 30, 60]:
        start_dt = np.datetime64(cutoff - timedelta(days=days))
        recent = df[df['t_dat'] >= start_dt]
        recent_count = recent.groupby('article_id').agg({'customer_id': 'count'})
        recent_count.columns = [f'sales_last_{days}d']
        recent_count = recent_count.reset_index()
        article_basic = article_basic.merge(recent_count, on='article_id', how='left')
        article_basic[f'sales_last_{days}d'] = article_basic[f'sales_last_{days}d'].fillna(0)
    
    article_basic['trend_7_14'] = article_basic['sales_last_7d'] / article_basic['sales_last_14d'].clip(lower=1)
    article_basic['trend_14_30'] = article_basic['sales_last_14d'] / article_basic['sales_last_30d'].clip(lower=1)
    article_basic['trend_30_60'] = article_basic['sales_last_30d'] / article_basic['sales_last_60d'].clip(lower=1)
    
    ua_count = df.groupby(['customer_id', 'article_id']).agg({'t_dat': 'count'})
    ua_count.columns = ['purchase_count']
    ua_count = ua_count.reset_index()
    ua_count['is_repeat'] = (ua_count['purchase_count'] > 1).astype('int32')
    article_repeat = ua_count.groupby('article_id').agg({
        'purchase_count': 'mean',
        'is_repeat': 'mean'
    })
    article_repeat.columns = ['avg_purchase_count_per_buyer', 'article_repeat_rate']
    article_repeat = article_repeat.reset_index()
    article_basic = article_basic.merge(article_repeat, on='article_id', how='left')
    article_basic['avg_purchase_count_per_buyer'] = article_basic['avg_purchase_count_per_buyer'].fillna(1.0)
    article_basic['article_repeat_rate'] = article_basic['article_repeat_rate'].fillna(0.0)
    
    art_attrs = articles_gdf[['article_id', 'product_type_no', 'garment_group_no',
                               'index_group_no', 'section_no', 'department_no', 'product_code']]
    article_basic = article_basic.merge(art_attrs, on='article_id', how='left')
    
    variant_count = articles_gdf.groupby('product_code').agg({'article_id': 'count'})
    variant_count.columns = ['variant_count']
    variant_count = variant_count.reset_index()
    article_basic = article_basic.merge(variant_count, on='product_code', how='left')
    article_basic['variant_count'] = article_basic['variant_count'].fillna(1)
    article_basic = article_basic.drop(columns=['product_code'])
    
    article_basic.to_pandas().to_parquet(str(cache_path))
    elapsed = time.time() - start
    print(f"  商品特徴量: {article_basic.shape} （{elapsed/60:.1f}分）保存完了")
    return article_basic


def build_user_article_features(transactions_gdf, articles_gdf, customers_gdf, cutoff, candidates_gdf):
    """ユーザー×商品特徴量（cuDF版）"""
    print("  ユーザー×商品特徴量作成中...")
    start = time.time()
    cutoff_dt = np.datetime64(cutoff)
    df = transactions_gdf[transactions_gdf['t_dat'] <= cutoff_dt]
    
    result = candidates_gdf.copy()
    
    source_names = ['timedecay', 'repurchase', 'variant', 'also_bought',
                    'item2item_cf', 'bin_popular', 'purchase_interval']
    for name in source_names:
        path = CANDIDATE_DIR / f"{name}_{cutoff}.parquet"
        if not path.exists():
            result[f'is_{name}'] = 0
            continue
        source_df = cudf.read_parquet(str(path), columns=['customer_id', 'article_id'])
        source_df = source_df.drop_duplicates()
        source_df[f'is_{name}'] = 1
        result = result.merge(source_df, on=['customer_id', 'article_id'], how='left')
        result[f'is_{name}'] = result[f'is_{name}'].fillna(0).astype('int32')
        print(f"    is_{name}: 完了")
    
    repeat_count = df.groupby(['customer_id', 'article_id']).agg({'t_dat': 'count'})
    repeat_count.columns = ['repeat_count']
    repeat_count = repeat_count.reset_index()
    result = result.merge(repeat_count, on=['customer_id', 'article_id'], how='left')
    result['repeat_count'] = result['repeat_count'].fillna(0)
    
    last_purchase_article = df.groupby(['customer_id', 'article_id']).agg({'t_dat': 'max'})
    last_purchase_article.columns = ['last_purchase_date']
    last_purchase_article = last_purchase_article.reset_index()
    last_purchase_article['days_since_last_purchase_article'] = (
        cutoff_dt - last_purchase_article['last_purchase_date']
    ).dt.days
    last_purchase_article = last_purchase_article.drop(columns=['last_purchase_date'])
    result = result.merge(last_purchase_article, on=['customer_id', 'article_id'], how='left')
    result['days_since_last_purchase_article'] = result['days_since_last_purchase_article'].fillna(-1)
    
    df_age = df.merge(customers_gdf[['customer_id', 'age_bin']], on='customer_id', how='left')
    age_bin_counts = df_age.groupby(['age_bin', 'article_id']).agg({'customer_id': 'count'})
    age_bin_counts.columns = ['age_bin_article_count']
    age_bin_counts = age_bin_counts.reset_index()
    age_bin_totals = df_age.groupby('age_bin').agg({'customer_id': 'count'})
    age_bin_totals.columns = ['age_bin_total']
    age_bin_totals = age_bin_totals.reset_index()
    age_bin_rate = age_bin_counts.merge(age_bin_totals, on='age_bin', how='left')
    age_bin_rate['age_bin_purchase_rate'] = age_bin_rate['age_bin_article_count'] / age_bin_rate['age_bin_total']
    age_bin_rate = age_bin_rate[['age_bin', 'article_id', 'age_bin_purchase_rate']]
    result = result.merge(customers_gdf[['customer_id', 'age_bin']], on='customer_id', how='left')
    result = result.merge(age_bin_rate, on=['age_bin', 'article_id'], how='left')
    result['age_bin_purchase_rate'] = result['age_bin_purchase_rate'].fillna(0)
    result = result.drop(columns=['age_bin'])
    
    art_cats = articles_gdf[['article_id', 'product_type_no', 'garment_group_no', 'index_group_no']]
    df_cat = df.merge(art_cats, on='article_id', how='left')
    user_total = df.groupby('customer_id').agg({'article_id': 'count'})
    user_total.columns = ['user_total']
    user_total = user_total.reset_index()
    
    for cat_col in ['product_type_no', 'garment_group_no', 'index_group_no']:
        user_cat = df_cat.groupby(['customer_id', cat_col]).agg({'article_id': 'count'})
        user_cat.columns = [f'user_{cat_col}_count']
        user_cat = user_cat.reset_index()
        user_cat = user_cat.merge(user_total, on='customer_id', how='left')
        user_cat[f'user_{cat_col}_ratio'] = user_cat[f'user_{cat_col}_count'] / user_cat['user_total']
        user_cat = user_cat.drop(columns=['user_total'])
        art_cat_col = articles_gdf[['article_id', cat_col]]
        result = result.merge(art_cat_col, on='article_id', how='left')
        result = result.merge(user_cat, on=['customer_id', cat_col], how='left')
        result[f'user_{cat_col}_count'] = result[f'user_{cat_col}_count'].fillna(0)
        result[f'user_{cat_col}_ratio'] = result[f'user_{cat_col}_ratio'].fillna(0)
        result = result.drop(columns=[cat_col])
        print(f"    {cat_col}: 完了")
    
    for cat_col in ['product_type_no', 'garment_group_no']:
        last_cat = df_cat.groupby(['customer_id', cat_col]).agg({'t_dat': 'max'})
        last_cat.columns = ['last_cat_date']
        last_cat = last_cat.reset_index()
        last_cat[f'days_since_last_{cat_col}'] = (cutoff_dt - last_cat['last_cat_date']).dt.days
        last_cat = last_cat.drop(columns=['last_cat_date'])
        art_cat_col = articles_gdf[['article_id', cat_col]]
        result = result.merge(art_cat_col, on='article_id', how='left')
        result = result.merge(last_cat, on=['customer_id', cat_col], how='left')
        result[f'days_since_last_{cat_col}'] = result[f'days_since_last_{cat_col}'].fillna(-1)
        result = result.drop(columns=[cat_col])
        print(f"    days_since_last_{cat_col}: 完了")
    
    elapsed = time.time() - start
    print(f"  ユーザー×商品特徴量: {result.shape} （{elapsed/60:.1f}分）")
    return result


def build_all_features(transactions_gdf, articles_gdf, customers_gdf, cutoff, candidates_gdf):
    """全特徴量を結合（キャッシュ付き）"""
    print(f"\n特徴量作成開始（cutoff={cutoff}）...")
    
    user_feat = build_user_features(transactions_gdf, customers_gdf, cutoff)
    article_feat = build_article_features(transactions_gdf, articles_gdf, cutoff)
    ua_feat = build_user_article_features(transactions_gdf, articles_gdf, customers_gdf, cutoff, candidates_gdf)
    
    result = ua_feat
    result = result.merge(user_feat, on='customer_id', how='left')
    result = result.merge(article_feat, on='article_id', how='left')
    
    for col in result.columns:
        if result[col].dtype in ['float32', 'float64']:
            result[col] = result[col].fillna(0.0)
        elif result[col].dtype in ['int32', 'int64']:
            result[col] = result[col].fillna(-1)
    
    print(f"\n特徴量作成完了: {result.shape}")
    return result

print("セル2完了!")

セル2完了!


In [15]:
from catboost import CatBoost, Pool

# 候補スコア統合
print(f"学習データ作成中（cutoff={TRAIN_CUTOFF}）...")
train_actuals = get_actuals(transactions, TRAIN_CUTOFF)
train_customers = list(train_actuals.keys())
print(f"  train_customers: {len(train_customers):,}")

train_scored = (
    create_candidates_scored(TRAIN_CUTOFF, active_logics, train_customers)
    .filter(pl.col('rank') <= N_CANDIDATES)
)
actuals_df = pl.DataFrame([
    {'customer_id': int(cid), 'article_id': int(aid), 'label': 1}
    for cid, aids in train_actuals.items()
    for aid in aids
]).with_columns([pl.col('customer_id').cast(pl.Int32), pl.col('article_id').cast(pl.Int32)])

train_scored = train_scored.join(actuals_df, on=['customer_id', 'article_id'], how='left').with_columns(
    pl.col('label').fill_null(0)
)
print(f"  候補数: {len(train_scored):,} / 正例数: {train_scored['label'].sum():,}")

# 特徴量作成
train_candidates_gdf = cudf.from_pandas(train_scored.to_pandas())
train_features_gdf = build_all_features(
    transactions_gdf, articles_gdf, customers_gdf, TRAIN_CUTOFF, train_candidates_gdf
)

# pandas変換・学習
train_df = train_features_gdf.to_pandas()
exclude_cols = ['customer_id', 'article_id', 'candidate_score', 'rank', 'label']
feature_cols = [c for c in train_df.columns if c not in exclude_cols]
print(f"特徴量数: {len(feature_cols)}")

# feature_cols保存
with open(str(MODEL_DIR / 'feature_cols.json'), 'w') as f:
    json.dump(feature_cols, f)

train_df_sorted = train_df.sort_values('customer_id')
train_pool = Pool(
    data=train_df_sorted[feature_cols],
    label=train_df_sorted['label'],
    group_id=train_df_sorted['customer_id']
)

model = CatBoost({
    'loss_function': 'YetiRank',
    'iterations': 300,
    'learning_rate': 0.05,
    'depth': 6,
    'verbose': 50,
    'early_stopping_rounds': 30,
})
model.fit(train_pool)
print("学習完了!")

# モデル保存
model.save_model(str(MODEL_DIR / 'catboost_model.cbm'))
print(f"モデル保存: {MODEL_DIR / 'catboost_model.cbm'}")

# 特徴量重要度
importance = model.get_feature_importance(train_pool, type='PredictionValuesChange')
feat_imp = sorted(zip(feature_cols, importance), key=lambda x: -x[1])
print("\n【特徴量重要度 TOP20】")
for i, (name, imp) in enumerate(feat_imp[:20], 1):
    print(f"  {i:>2}. {name:<35} {imp:.4f}")

# メモリ解放
del train_candidates_gdf, train_features_gdf, train_df, train_df_sorted, train_pool
gc.collect()
print("\nセル3完了!")

学習データ作成中（cutoff=2020-09-08）...
  train_customers: 72,019
  候補数: 6,574,651 / 正例数: 24,062

特徴量作成開始（cutoff=2020-09-08）...
  ユーザー特徴量作成中...
  ユーザー特徴量: (1351314, 22) （7.0分）保存完了
  商品特徴量作成中...
  商品特徴量: (102967, 19) （0.1分）保存完了
  ユーザー×商品特徴量作成中...
    is_timedecay: 完了
    is_repurchase: 完了
    is_variant: 完了
    is_also_bought: 完了
    is_item2item_cf: 完了
    is_bin_popular: 完了
    is_purchase_interval: 完了
    product_type_no: 完了
    garment_group_no: 完了
    index_group_no: 完了
    days_since_last_product_type_no: 完了
    days_since_last_garment_group_no: 完了
  ユーザー×商品特徴量: (6574651, 23) （0.4分）

特徴量作成完了: (6574651, 62)
特徴量数: 57
0:	total: 1.2s	remaining: 5m 58s
50:	total: 54.2s	remaining: 4m 24s
100:	total: 1m 45s	remaining: 3m 27s
150:	total: 2m 35s	remaining: 2m 33s
200:	total: 3m 26s	remaining: 1m 41s
250:	total: 4m 18s	remaining: 50.4s
299:	total: 5m 7s	remaining: 0us
学習完了!
モデル保存: /home/rapids/models/catboost_model.cbm

【特徴量重要度 TOP20】
   1. sales_last_7d                       16.6089
   2. is_timede

In [16]:
# モデルが消えている場合はロード
try:
    model
except NameError:
    from catboost import CatBoost
    model = CatBoost()
    model.load_model(str(MODEL_DIR / 'catboost_model.cbm'))
    with open(str(MODEL_DIR / 'feature_cols.json'), 'r') as f:
        feature_cols = json.load(f)
    print("モデルをロードしました")

print(f"valid評価中（cutoff={VALID_CUTOFF}）...")
valid_actuals = get_actuals(transactions, VALID_CUTOFF)
valid_customers = list(valid_actuals.keys())
print(f"  valid_customers: {len(valid_customers):,}")

valid_scored = (
    create_candidates_scored(VALID_CUTOFF, active_logics, valid_customers)
    .filter(pl.col('rank') <= N_CANDIDATES)
)
valid_actuals_df = pl.DataFrame([
    {'customer_id': int(cid), 'article_id': int(aid), 'label': 1}
    for cid, aids in valid_actuals.items()
    for aid in aids
]).with_columns([pl.col('customer_id').cast(pl.Int32), pl.col('article_id').cast(pl.Int32)])

valid_scored = valid_scored.join(valid_actuals_df, on=['customer_id', 'article_id'], how='left').with_columns(
    pl.col('label').fill_null(0)
)
print(f"  候補数: {len(valid_scored):,} / 正例数: {valid_scored['label'].sum():,}")

valid_candidates_gdf = cudf.from_pandas(valid_scored.to_pandas())
valid_features_gdf = build_all_features(
    transactions_gdf, articles_gdf, customers_gdf, VALID_CUTOFF, valid_candidates_gdf
)

valid_df = valid_features_gdf.to_pandas()
valid_df['pred_score'] = model.predict(valid_df[feature_cols])

metric_acc = {k: {'recall': [], 'map': []} for k in EVAL_CUTOFFS}
for cid, group in valid_df.groupby('customer_id'):
    actual_set = valid_actuals.get(cid, set())
    if not actual_set:
        continue
    prediction = group.sort_values('pred_score', ascending=False)['article_id'].tolist()
    for k in EVAL_CUTOFFS:
        hits = len(set(prediction[:k]) & actual_set)
        metric_acc[k]['recall'].append(hits / len(actual_set))
        metric_acc[k]['map'].append(apk(actual_set, prediction, k))

print("\n" + "="*50)
print("【valid評価】")
print("="*50)
print(f"{'k':>6} {'recall':>10} {'MAP':>10}")
print("-"*30)
for k in EVAL_CUTOFFS:
    recall = np.mean(metric_acc[k]['recall'])
    map_k = np.mean(metric_acc[k]['map'])
    print(f"{k:>6} {recall:>10.6f} {map_k:>10.6f}")

print(f"\n【候補生成のみとの比較】")
print(f"  候補生成のみ MAP@12: 0.028827")
print(f"  リランキング MAP@12: {np.mean(metric_acc[12]['map']):.6f}")

# メモリ解放
del valid_candidates_gdf, valid_features_gdf, valid_df
gc.collect()
print("\nセル4完了!")

valid評価中（cutoff=2020-09-15）...
  valid_customers: 68,984
  候補数: 6,274,713 / 正例数: 23,518

特徴量作成開始（cutoff=2020-09-15）...
  ユーザー特徴量作成中...
  ユーザー特徴量: (1356709, 22) （0.6分）保存完了
  商品特徴量作成中...
  商品特徴量: (103880, 19) （0.1分）保存完了
  ユーザー×商品特徴量作成中...
    is_timedecay: 完了
    is_repurchase: 完了
    is_variant: 完了
    is_also_bought: 完了
    is_item2item_cf: 完了
    is_bin_popular: 完了
    is_purchase_interval: 完了
    product_type_no: 完了
    garment_group_no: 完了
    index_group_no: 完了
    days_since_last_product_type_no: 完了
    days_since_last_garment_group_no: 完了
  ユーザー×商品特徴量: (6274713, 23) （0.4分）

特徴量作成完了: (6274713, 62)

【valid評価】
     k     recall        MAP
------------------------------
    12   0.071052   0.034213
    24   0.093948   0.035725
    48   0.121444   0.036690
    96   0.133132   0.036940
   120   0.133147   0.036940

【候補生成のみとの比較】
  候補生成のみ MAP@12: 0.028827
  リランキング MAP@12: 0.034213

セル4完了!


In [4]:
# ============================================================
# 再学習（cutoff=2020-09-15）
# ============================================================
from catboost import CatBoost, Pool

RETRAIN_CUTOFF = date(2020, 9, 15)
print(f"再学習（cutoff={RETRAIN_CUTOFF}）...")

# STEP1: 候補スコア統合
print("\n[STEP1] 候補スコア統合中...")
retrain_actuals = get_actuals(transactions, RETRAIN_CUTOFF)
retrain_customers = list(retrain_actuals.keys())
print(f"  train_customers: {len(retrain_customers):,}")

retrain_scored = (
    create_candidates_scored(RETRAIN_CUTOFF, active_logics, retrain_customers)
    .filter(pl.col('rank') <= N_CANDIDATES)
)
actuals_df = pl.DataFrame([
    {'customer_id': int(cid), 'article_id': int(aid), 'label': 1}
    for cid, aids in retrain_actuals.items()
    for aid in aids
]).with_columns([pl.col('customer_id').cast(pl.Int32), pl.col('article_id').cast(pl.Int32)])

retrain_scored = retrain_scored.join(actuals_df, on=['customer_id', 'article_id'], how='left').with_columns(
    pl.col('label').fill_null(0)
)
print(f"  候補数: {len(retrain_scored):,} / 正例数: {retrain_scored['label'].sum():,}")

# 候補スコアを保存
retrain_scored.write_parquet(str(FEATURE_DIR / f'retrain_scored_{RETRAIN_CUTOFF}.parquet'))
print(f"  保存完了: retrain_scored_{RETRAIN_CUTOFF}.parquet")
print("[STEP1] 完了!")

再学習（cutoff=2020-09-15）...

[STEP1] 候補スコア統合中...
  train_customers: 68,984
  候補数: 6,274,713 / 正例数: 23,523
  保存完了: retrain_scored_2020-09-15.parquet
[STEP1] 完了!


In [5]:
# STEP2: 特徴量作成
print("\n[STEP2] 特徴量作成中...")

# キャッシュが消えている場合は再読み込み
retrain_scored_path = FEATURE_DIR / f'retrain_scored_{RETRAIN_CUTOFF}.parquet'
if 'retrain_scored' not in dir():
    retrain_scored = pl.read_parquet(str(retrain_scored_path))
    print(f"  retrain_scored読み込み: {len(retrain_scored):,}行")

retrain_candidates_gdf = cudf.from_pandas(retrain_scored.to_pandas())
retrain_features_gdf = build_all_features(
    transactions_gdf, articles_gdf, customers_gdf, RETRAIN_CUTOFF, retrain_candidates_gdf
)

# 特徴量を保存
retrain_features_gdf.to_pandas().to_parquet(str(FEATURE_DIR / f'retrain_features_{RETRAIN_CUTOFF}.parquet'))
print(f"  保存完了: retrain_features_{RETRAIN_CUTOFF}.parquet")
print("[STEP2] 完了!")


[STEP2] 特徴量作成中...

特徴量作成開始（cutoff=2020-09-15）...
  ユーザー特徴量: キャッシュ読み込み /home/rapids/features/user_features_2020-09-15.parquet
  商品特徴量: キャッシュ読み込み /home/rapids/features/article_features_2020-09-15.parquet
  ユーザー×商品特徴量作成中...
    is_timedecay: 完了
    is_repurchase: 完了
    is_variant: 完了
    is_also_bought: 完了
    is_item2item_cf: 完了
    is_bin_popular: 完了
    is_purchase_interval: 完了
    product_type_no: 完了
    garment_group_no: 完了
    index_group_no: 完了
    days_since_last_product_type_no: 完了
    days_since_last_garment_group_no: 完了
  ユーザー×商品特徴量: (6274713, 23) （0.5分）

特徴量作成完了: (6274713, 62)
  保存完了: retrain_features_2020-09-15.parquet
[STEP2] 完了!


In [ ]:
# STEP3: CatBoost学習
print("\n[STEP3] CatBoost学習中...")

# 特徴量が消えている場合は再読み込み
retrain_feat_path = FEATURE_DIR / f'retrain_features_{RETRAIN_CUTOFF}.parquet'
if 'retrain_features_gdf' not in dir():
    retrain_df = pd.read_parquet(str(retrain_feat_path))
    print(f"  retrain_features読み込み: {len(retrain_df):,}行")
else:
    retrain_df = retrain_features_gdf.to_pandas()
    del retrain_features_gdf
    gc.collect()

# feature_cols読み込み
with open(str(MODEL_DIR / 'feature_cols.json'), 'r') as f:
    feature_cols = json.load(f)
print(f"  特徴量数: {len(feature_cols)}")

retrain_df_sorted = retrain_df.sort_values('customer_id')
retrain_pool = Pool(
    data=retrain_df_sorted[feature_cols],
    label=retrain_df_sorted['label'],
    group_id=retrain_df_sorted['customer_id']
)

model_v2 = CatBoost({
    'loss_function': 'YetiRank',
    'iterations': 300,
    'learning_rate': 0.05,
    'depth': 6,
    'verbose': 50,
    'early_stopping_rounds': 30,
})
model_v2.fit(retrain_pool)
print("学習完了!")

# モデル保存
model_v2.save_model(str(MODEL_DIR / 'catboost_model_v2.cbm'))
print(f"モデル保存: {MODEL_DIR / 'catboost_model_v2.cbm'}")

# 特徴量重要度
importance = model_v2.get_feature_importance(retrain_pool, type='PredictionValuesChange')
feat_imp = sorted(zip(feature_cols, importance), key=lambda x: -x[1])
print("\n【特徴量重要度 TOP20】")
for i, (name, imp) in enumerate(feat_imp[:20], 1):
    print(f"  {i:>2}. {name:<35} {imp:.4f}")

del retrain_df, retrain_df_sorted, retrain_pool
gc.collect()
print("\n[STEP3] 完了!")


[STEP3] CatBoost学習中...
  特徴量数: 57
0:	total: 1.09s	remaining: 5m 25s
50:	total: 51.3s	remaining: 4m 10s
100:	total: 1m 42s	remaining: 3m 20s
150:	total: 2m 29s	remaining: 2m 27s
200:	total: 3m 16s	remaining: 1m 37s
250:	total: 4m 4s	remaining: 47.7s
299:	total: 4m 52s	remaining: 0us
学習完了!
モデル保存: /home/rapids/models/catboost_model_v2.cbm

【特徴量重要度 TOP20】
   1. sales_last_7d                       13.7805
   2. is_timedecay                        7.8139
   3. days_since_last_purchase_article    7.2985
   4. trend_30_60                         4.8595
   5. sales_last_14d                      4.7200
   6. sales_last_30d                      4.3833
   7. user_index_group_no_ratio           3.9097
   8. days_since_last_garment_group_no    3.7564
   9. purchases_last_7d                   3.3241
  10. is_variant                          2.8852
  11. user_garment_group_no_ratio         2.7144
  12. user_product_type_no_ratio          2.6645
  13. trend_14_30                         2.4722
  14. d

: 

In [6]:
# STEP4: 提出ファイル作成（メモリ軽量版）
print("\n[STEP4] 提出ファイル作成中...")

from catboost import CatBoost

# モデルロード
try:
    model_v2
except NameError:
    model_v2 = CatBoost()
    model_v2.load_model(str(MODEL_DIR / 'catboost_model_v2.cbm'))
    with open(str(MODEL_DIR / 'feature_cols.json'), 'r') as f:
        feature_cols = json.load(f)
    print("  モデルをロードしました")

SUB_CUTOFF = date(2020, 9, 22)
print(f"  提出用cutoff: {SUB_CUTOFF}")

# スコア統合済みファイル
scored_df = pl.read_parquet(str(CANDIDATE_DIR / f'candidates_scored_{SUB_CUTOFF}.parquet'))
print(f"  全候補数: {len(scored_df):,}行")

# ユーザー・商品特徴量（キャッシュ）
user_feat_pd = build_user_features(transactions_gdf, customers_gdf, SUB_CUTOFF).to_pandas()
article_feat_pd = build_article_features(transactions_gdf, articles_gdf, SUB_CUTOFF).to_pandas()
gc.collect()

# 事前計算（Polarsで軽量に）
print("  事前計算中...")
df_train_pl = transactions.filter(pl.col('t_dat') <= pl.lit(SUB_CUTOFF))

repeat_count_pl = df_train_pl.group_by(['customer_id', 'article_id']).agg(
    pl.len().alias('repeat_count')
)

last_purchase_article_pl = (
    df_train_pl.group_by(['customer_id', 'article_id'])
    .agg(pl.col('t_dat').max().alias('last_date'))
    .with_columns(
        (pl.lit(SUB_CUTOFF) - pl.col('last_date')).dt.total_days().alias('days_since_last_purchase_article')
    )
    .drop('last_date')
)

customers_age_pl = customers.select(['customer_id', 'age_bin'])

df_age_pl = df_train_pl.join(customers_age_pl, on='customer_id', how='left')
age_bin_counts_pl = df_age_pl.group_by(['age_bin', 'article_id']).agg(pl.len().alias('cnt'))
age_bin_totals_pl = df_age_pl.group_by('age_bin').agg(pl.len().alias('total'))
age_bin_rate_pl = (
    age_bin_counts_pl.join(age_bin_totals_pl, on='age_bin', how='left')
    .with_columns((pl.col('cnt') / pl.col('total')).alias('age_bin_purchase_rate'))
    .select(['age_bin', 'article_id', 'age_bin_purchase_rate'])
)

art_cats_pl = articles.select(['article_id', 'product_type_no', 'garment_group_no', 'index_group_no'])
df_cat_pl = df_train_pl.join(art_cats_pl, on='article_id', how='left')
user_total_pl = df_train_pl.group_by('customer_id').agg(pl.len().alias('user_total'))

user_cat_pls = {}
for cat_col in ['product_type_no', 'garment_group_no', 'index_group_no']:
    uc = (
        df_cat_pl.group_by(['customer_id', cat_col]).agg(pl.len().alias(f'user_{cat_col}_count'))
        .join(user_total_pl, on='customer_id', how='left')
        .with_columns((pl.col(f'user_{cat_col}_count') / pl.col('user_total')).alias(f'user_{cat_col}_ratio'))
        .drop('user_total')
    )
    user_cat_pls[cat_col] = uc

last_cat_pls = {}
for cat_col in ['product_type_no', 'garment_group_no']:
    lc = (
        df_cat_pl.group_by(['customer_id', cat_col])
        .agg(pl.col('t_dat').max().alias('last_cat_date'))
        .with_columns(
            (pl.lit(SUB_CUTOFF) - pl.col('last_cat_date')).dt.total_days().alias(f'days_since_last_{cat_col}')
        )
        .drop('last_cat_date')
    )
    last_cat_pls[cat_col] = lc

# 候補ソースフラグ（Polarsで軽量に）
source_names = ['timedecay', 'repurchase', 'variant', 'also_bought',
                'item2item_cf', 'bin_popular', 'purchase_interval']
source_pls = {}
for name in source_names:
    path = CANDIDATE_DIR / f"{name}_{SUB_CUTOFF}.parquet"
    if path.exists():
        source_pls[name] = (
            pl.read_parquet(path, columns=['customer_id', 'article_id'])
            .unique()
            .with_columns(pl.lit(1).alias(f'is_{name}'))
        )
    else:
        source_pls[name] = None

del df_train_pl, df_age_pl, df_cat_pl
gc.collect()
print("  事前計算完了!")

# バッチ処理
BATCH_SIZE = 100000
all_customer_ids = scored_df['customer_id'].unique().to_list()
all_predictions = {}

# 途中再開用：既に予測済みのバッチをスキップ
resume_path = Path('/home/rapids/predictions_partial.json')

for batch_start in range(0, len(all_customer_ids), BATCH_SIZE):
    batch_cids = all_customer_ids[batch_start:batch_start+BATCH_SIZE]
    batch_end = min(batch_start + BATCH_SIZE, len(all_customer_ids))
    batch_num = batch_start // BATCH_SIZE + 1
    print(f"\n  バッチ {batch_num}: {batch_start:,}〜{batch_end:,}")
    
    # Polarsで候補を取得・特徴量結合
    batch = scored_df.filter(pl.col('customer_id').is_in(batch_cids))
    
    # 候補ソースフラグ
    for name in source_names:
        if source_pls[name] is not None:
            batch = batch.join(source_pls[name], on=['customer_id', 'article_id'], how='left').with_columns(
                pl.col(f'is_{name}').fill_null(0)
            )
        else:
            batch = batch.with_columns(pl.lit(0).alias(f'is_{name}'))
    
    # repeat_count
    batch = batch.join(repeat_count_pl, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('repeat_count').fill_null(0)
    )
    
    # days_since_last_purchase_article
    batch = batch.join(last_purchase_article_pl, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('days_since_last_purchase_article').fill_null(-1)
    )
    
    # age_bin_purchase_rate
    batch = batch.join(customers_age_pl, on='customer_id', how='left')
    batch = batch.join(age_bin_rate_pl, on=['age_bin', 'article_id'], how='left').with_columns(
        pl.col('age_bin_purchase_rate').fill_null(0)
    )
    batch = batch.drop('age_bin')
    
    # ユーザー×カテゴリ
    batch = batch.join(art_cats_pl, on='article_id', how='left')
    for cat_col in ['product_type_no', 'garment_group_no', 'index_group_no']:
        batch = batch.join(user_cat_pls[cat_col], on=['customer_id', cat_col], how='left').with_columns([
            pl.col(f'user_{cat_col}_count').fill_null(0),
            pl.col(f'user_{cat_col}_ratio').fill_null(0),
        ])
    
    for cat_col in ['product_type_no', 'garment_group_no']:
        batch = batch.join(last_cat_pls[cat_col], on=['customer_id', cat_col], how='left').with_columns(
            pl.col(f'days_since_last_{cat_col}').fill_null(-1)
        )
    
    batch = batch.drop(['product_type_no', 'garment_group_no', 'index_group_no'])
    
    # pandasに変換してユーザー・商品特徴量結合
    batch_pd = batch.to_pandas()
    del batch; gc.collect()
    
    batch_pd = batch_pd.merge(user_feat_pd, on='customer_id', how='left')
    batch_pd = batch_pd.merge(article_feat_pd, on='article_id', how='left')
    
    for col in feature_cols:
        if col in batch_pd.columns:
            batch_pd[col] = batch_pd[col].fillna(0)
    
    # 予測
    batch_pd['pred_score'] = model_v2.predict(batch_pd[feature_cols])
    
    # 上位12件
    top12 = (
        batch_pd
        .sort_values(['customer_id', 'pred_score'], ascending=[True, False])
        .groupby('customer_id')
        .head(12)
    )
    for cid, group in top12.groupby('customer_id'):
        arts = group['article_id'].tolist()
        all_predictions[cid] = ' '.join([article_id_reverse[a] for a in arts])
    
    del batch_pd, top12; gc.collect()
    print(f"    完了: {len(batch_cids):,}ユーザー（累計: {len(all_predictions):,}）")

# 全ユーザーにマージ
print(f"\n予測済みユーザー数: {len(all_predictions):,}")

bin_pop = pl.read_parquet(str(CANDIDATE_DIR / f'bin_popular_{SUB_CUTOFF}.parquet'))
fallback_arts = bin_pop.sort('rank').select('article_id').head(12).to_series().to_list()
fallback_str = ' '.join([article_id_reverse[a] for a in fallback_arts])

rows = []
for cid_str, cid_int in zip(customer_id_map.keys(), customer_id_map.values()):
    pred = all_predictions.get(cid_int, fallback_str)
    rows.append({'customer_id': cid_str, 'prediction': pred})

submission = pd.DataFrame(rows)
submission.to_csv('/workspace/outputs/submissions/submission_rerank_v5.csv', index=False)
print(f"\n保存完了: submission_rerank_v5.csv")
print(f"ユーザー数: {len(submission):,}")
print(submission.head(5))


[STEP4] 提出ファイル作成中...
  モデルをロードしました
  提出用cutoff: 2020-09-22
  全候補数: 122,600,347行
  ユーザー特徴量: キャッシュ読み込み /home/rapids/features/user_features_2020-09-22.parquet
  商品特徴量: キャッシュ読み込み /home/rapids/features/article_features_2020-09-22.parquet
  事前計算中...
  事前計算完了!

  バッチ 1: 0〜100,000
    完了: 100,000ユーザー（累計: 100,000）

  バッチ 2: 100,000〜200,000
    完了: 100,000ユーザー（累計: 200,000）

  バッチ 3: 200,000〜300,000
    完了: 100,000ユーザー（累計: 300,000）

  バッチ 4: 300,000〜400,000
    完了: 100,000ユーザー（累計: 400,000）

  バッチ 5: 400,000〜500,000
    完了: 100,000ユーザー（累計: 500,000）

  バッチ 6: 500,000〜600,000
    完了: 100,000ユーザー（累計: 600,000）

  バッチ 7: 600,000〜700,000
    完了: 100,000ユーザー（累計: 700,000）

  バッチ 8: 700,000〜800,000
    完了: 100,000ユーザー（累計: 800,000）

  バッチ 9: 800,000〜900,000
    完了: 100,000ユーザー（累計: 900,000）

  バッチ 10: 900,000〜1,000,000
    完了: 100,000ユーザー（累計: 1,000,000）

  バッチ 11: 1,000,000〜1,100,000
    完了: 100,000ユーザー（累計: 1,100,000）

  バッチ 12: 1,100,000〜1,200,000
    完了: 100,000ユーザー（累計: 1,200,000）

  バッチ 13: 1,200,000〜1,3